# TP3: Hello Wolrd Transformers, Adriana RIZK

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel, pipeline

/Users/adrianarizk/Desktop/S5/NLP/tp3_hello_world_transformers/nlp_course/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Va chercher sur HuggingFace le tokenizer correspondant à DistilBERT

* Charge les poids du modèle DistilBERT

* Ces poids contiennent ce que le modèle a appris sur des milliards de mots

In [4]:
model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

* input_ids: la phrase convertie en IDs numériques

* attention_mask: 1 = lire le token, 0 = ignorer (padding)

In [5]:
text = "Transformers are amazing for NLP."

tokens = tokenizer(text)
tokens

{'input_ids': [101, 19081, 2024, 6429, 2005, 17953, 2361, 1012, 102], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1]}

Le texte est normalisé en minuscules car c’est uncased

In [6]:
decoded = tokenizer.decode(tokens["input_ids"])
decoded

'[CLS] transformers are amazing for nlp. [SEP]'

## Question 1: Understanding Pipelines

Le pipeline charge automatiquement un modèle entraîné sur les sentiments.

In [7]:
classifier = pipeline("sentiment-analysis")
classifier("I love machine learning!")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use mps:0


[{'label': 'POSITIVE', 'score': 0.9998431205749512}]

--> Le modèle est 99.98% sûr que la phrase est positive

Tu n’as pas besoin de gérer la tokenisation, padding, modèle --> tout est fait automatiquement.

## Question 2: Text Classification Deep Dive

In [9]:
classifier = pipeline("sentiment-analysis")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use mps:0


**What dataset was this model fine-tuned on? What kind of text does it work best with?**

This model was fine-tuned on the dataset:

SST-2 (Stanford Sentiment Treebank v2)

This dataset contains short movie-review sentences labelled positive or negative

The model works best on short English sentences expressing an opinion, such as reviews, comments, tweets, etc

**The output includes a score field. What does this score represent? What range can it have?**

The score corresponds to:

--> the model’s confidence so the probability for the predicted label

It is the value after applying a softmax.

The score always lies between 0 and 1, 0 means not confident and 1 = very confident

**Challenge: Find a different text-classification model that classifies emotions (not just POS/NEG). What is its name?**

One example of an emotion classifier on the HuggingFace Hub is:

-->  j-hartmann/emotion-english-distilroberta-base

or another simpler model:

--> bhadresh-savani/distilbert-base-uncased-emotion

These models classify emotions such as joy, anger, fear, sadness, surprise, disgust, etcetera, not only positive/negative

In [11]:
emotion_model = pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base"
)

emotion_model("I feel so anxious but also excited.")

Device set to use mps:0


[{'label': 'fear', 'score': 0.9913376569747925}]

## Question 3: Named Entity Recognition (NER)

Name Entity Recognition

In [15]:
import pandas as pd
ner_tagger = pipeline("ner", aggregation_strategy="simple")
outputs = ner_tagger(text)
pd.DataFrame(outputs)    

No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision 4c53496 (https://huggingface.co/dbmdz/bert-large-cased-finetuned-conll03-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Some weights of the model checkpoint at dbmdz/bert-large-cased-finetuned-conll03-english were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use mps:0


,entity_group,score,word,start,end
0,ORG,0.98887,NLP,29,32


**1. What does the aggregation_strategy="simple" parameter do?**

aggregation_strategy="simple" tells the pipeline to:

--> merge tokens which belong to the same entity into one single prediction

For example:

* Without aggregation --> we have New, York , it appears separately

* Now, with aggregation --> we have New York that becomes one entity

It makes the output easier to read

**2. Looking at the output, what do the entity types mean?**

Common NER tags (CoNLL-2003 format):

Tag	Meaning
ORG	Organization (Amazon, UN, Google…)
LOC	Location (Germany, Europe…)
PER	Person (John, Merkel…)
MISC	Miscellaneous named entity

So the model identifies names, places, companies, brands, etc

**3. Why do some words appear with ## prefix (like ##tron and ##icons)? What does this tell you?**

The ## means the word was split by the tokenizer into subwords using WordPiece

For example:

* “Megatron” → Mega + ##tron

* “Decepticons” → De + ##cept + ##icons

This tells us that the tokenizer uses subword tokenization to handle rare or unknown words, it processes only tokens from its pre-defined vocabulary

**4. Why did the model split “Megatron” and “Decepticons” incorrectly? What does it tell you about its training data?**

It is beecause these words didn’t exist in the training dataset (CoNLL-2003)

--> the model doesn’t know Transformers robots, so it treats them as rare or unknown words, and splits them into smaller subwords (WordPiece)

The model performs best on news articles, people, companies, political locations, not sci-fi vocabulary

**5. Challenge: What is the CoNLL-2003 dataset?**

The model we used:

--> dbmdz/bert-large-cased-finetuned-conll03-english

is trained on the: CoNLL-2003 NER dataset

This dataset contains news articles from Reuters which are nnotated with four entity types: PER, ORG, LOC, MISC
It is the standard benchmark for English NER

## Question 4: Question Answering Systems

In [16]:
reader = pipeline("question-answering")

No model was supplied, defaulted to distilbert/distilbert-base-cased-distilled-squad and revision 564e9b5 (https://huggingface.co/distilbert/distilbert-base-cased-distilled-squad).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use mps:0


**1. What type of question answering is this? (Extractive vs Generative)**
This is an extractive QA model.

* Extractive QA = the model doesn't generate new text
* it only selects a span of text from the context
* And the answer is always a substring of the input

This is extractive question answering

**2. The model outputs start and end indices. What do these represent? Why are they important?**

start and end are:

-->  the positions (token indices) where the answer begins and ends in the context

They are important because : 

* They allow the model to locate the exact span of text
* Without them, the model wouldn’t know what part of the context is the answer
* Extractive QA = basically predicting start and end positions

**3. What is the SQuAD dataset?**

SQuAD = Stanford Question Answering Dataset
Used to train and benchmark extractive QA models

Its characteritsics:

- real paragraphs from Wikipedia

- human-written questions

- answers are spans extracted directly from the paragraph

- used to fine-tune models like distilbert-base-cased-distilled-squad

So actually,  it measures how well a model can find the correct span in a text

**4. Think of a question this model CANNOT answer based on the text. Why would it fail?**

Example of a question it cannot answer:

- “Who is the CEO of the FIA right now?”
- “What year was ESSEC invented?”
- “Compute is 25 × 14 ?”

It fails because:

- The answer is not written in the context
- Extractive QA cannot invent new information
- It cannot reason or do math
- It can ONLY select text already present in the paragraph

**5. Challenge: Difference between extractive and generative QA + example model**

**Extractive QA**

- Selects part of the input
- cannot invent information

As examples:

- distilbert-base-cased-distilled-squad

- bert-large-uncased-whole-word-masking-finetuned-squad

**Generative QA**

- itses encoder-decoder or decoder-only models
- it generates an answer word by word
- it can answer open-ended questions and is not limited to the input text

As examples:

- google/flan-t5-base
- google/t5-small
- facebook/bart-large
- mistral-7b-instruct

Generative models behave more like ChatGPT: they produce new text, not just extract it

## Question 5: Text Summarization

**1. Difference between extractive and abstractive summarization**

- Extractive summarization: the model takes sentences or small parts directly from the text. It basically selects the most important lines without changing anything

- Abstractive summarization: the model rewrites the text in its own words. It creates a shorter version using new sentences, a bit like how a person would explain it

--> The pipeline in HuggingFace uses abstractive summarization

**2. Default model for summarization**
the model that gets loaded is:

-->  sshleifer/distilbart-cnn-12-6

It is abstractive, because it's based on BART which is a text-generation model. The architecture is encoder–decoder (a seq-to-seq model). It was trained on the CNN/DailyMail news dataset, which contains news articles and short human-written summaries.
So it works best on news-like paragraphs or short documents.

In [17]:
summarizer = pipeline("summarization")


No model was supplied, defaulted to sshleifer/distilbart-cnn-12-6 and revision a4f8f3e (https://huggingface.co/sshleifer/distilbart-cnn-12-6).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use mps:0


**3. What do max_length and min_length control? What if min_length > max_length?**

- max_length = the longest summary the model is allowed to produce

- min_length = the shortest summary the model is allowed to produce

If min_length is bigger than max_length, the model won’t know what to do, because the conditions contradict each other
Usually it causes an error or a bad output

**4. What does clean_up_tokenization_spaces=True do? Why is it useful?**

This option simply “cleans” the text by fixing extra spaces. For example, it removes strange spaces before commas or periods. It’s useful because summarization models sometimes generate text with awkward spacing, so this makes the final summary look normal.

**5. Challenge: Two summarization models**

- ***Model 1 : Good for short texts***

- Example: facebook/bart-large-cnn

- Works well for news articles

- Trained on CNN/DailyMail

- Makes short summaries (a few sentences)



- ***Model 2 : Good for long documents***

- Example: allenai/led-base-16384

- Can handle very long texts (reports, long articles)

- LED = Longformer Encoder Decoder

- Designed specifically for long documents

- Another option is Pegasus, which is also strong on longer texts.

In [18]:
outputs = summarizer(text, max_length=45, clean_up_tokenization_spaces=True)
print(outputs[0]['summary_text'])

Your min_length=56 must be inferior than your max_length=45.
Your max_length is set to 45, but your input_length is only 9. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=4)
/Users/adrianarizk/Desktop/S5/NLP/tp3_hello_world_transformers/nlp_course/lib/python3.12/site-packages/transformers/generation/utils.py:1633: UserWarning: Unfeasible length constraints: `min_length` (56) is larger than the maximum possible length (45). Generation will stop at the defined maximum length. You should decrease the minimum length and/or increase the maximum length.
  warnings.warn(


 Transformers are amazing for NLP. Transformers were amazing for us to use in our NLP toolkit. NLP is a toolkit that teaches us how to use the language of language in a new way.


## Question 6: Machine Translation

**1. Architecture behind the Helsinki-NLP/opus-mt-en-de model**

This model is based on MarianMT, which is a fast sequence-to-sequence architecture used for translation

* “OPUS” means “Open Parallel Corpus”
--> it's basically a big collection of translated texts used to train translation models

* “MT” = “Machine Translation”

So the name means: a translation model trained on OPUS data, for English to German

**2. How to find a model for English to French translation**

I would go on the Hugging Face Model Hub and simply search “English French translation”
Two common models are:

- Helsinki-NLP/opus-mt-en-fr (MarianMT, specific to English→French)

- facebook/m2m100_418M (multilingual, works for many language pairs)

(another option is Helsinki-NLP/opus-mt-fr-en in the other direction)

So there are several choices depending on what we want

**3. Difference between bilingual and multilingual translation models**

***Bilingual models***

- They translate one pair of languages only (ex: EN→FR)

- they're usually more accurate for that specific pair

- But we need one model for each direction
(ex: another model for FR→EN)

***Multilingual models***

- they cn translate many language pairs inside one model

- they are flexible and easier to use

- but sometimes a little less accurate than a bilingual model fine-tuned only on one direction

So it’s a trade-off:
-->  bilingual = very specialized

--> multilingual = more general

**4. Why do we specify "translation_en_to_de" in the code?**

we tell the pipeline exactly which direction we want to translate (English to German)
This is important because MarianMT models are trained per language pair, so the pipeline needs to load the right one

In [19]:
translator = pipeline("translation_en_to_de")

No model was supplied, defaulted to google-t5/t5-base and revision a9723ea (https://huggingface.co/google-t5/t5-base).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use mps:0


**5. What is the “sacremoses” library used for?**

The warning says that sacremoses is missing. This library is used for tokenization and detokenization (basically splitting text into words and putting it back together correctly).
Older translation models depend on it because they follow more traditional preprocessing steps.

**6. Challenge : A multilingual model + how many language pairs it supports**

A good multilingual model is:

-->  facebook/m2m100_418M

It can translate between any pair of the supported languages (100 languages total).This means there are thousands of possible translation directions.

Another example is:

--> facebook/mbart-large-50-many-to-many-mmt

This one supports 50 languages, and also allows many-to-many translation. These models are useful when we need flexibility or many different language combinations.

In [20]:
outputs = translator(text, clean_up_tokenization_spaces=True, min_length=100)
print(outputs[0]['translation_text'])

Transformers sind erstaunlich für NLP. Sie sind es, die es erlauben, es zu erlernen, es zu erlernen, es zu erlernen, es zu erlernen, es zu erlernen, es zu erlernen, es zu erlernen, es zu erlernen, es zu erlernen, es zu er


Fr --> Eng

In [2]:
from transformers import pipeline

translator = pipeline("translation", model="Helsinki-NLP/opus-mt-fr-en")

text = "Adriana est la reine du monde"
outputs = translator(text, clean_up_tokenization_spaces=True, min_length=10)

print(outputs[0]['translation_text'])

Device set to use mps:0


Adriana is the queen of the world.


## Question 7: Text Generation

**1. Default model used + GPT-2 characteristics**

the default model is: gpt2

Here are the points they ask for:

- Architecture: GPT-2 is a decoder-only Transformer. It doesn’t have an encoder. It only predicts the next token

- Number of parameters: The base GPT-2 model has 124 million parameters

- Type of generation: GPT-2 is autoregressive. This means it generates text one token at a time, and each new token depends on the previous ones

In [21]:
generator = pipeline("text-generation")

No model was supplied, defaulted to openai-community/gpt2 and revision 607a30d (https://huggingface.co/openai-community/gpt2).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use mps:0


**2. Why do we use set_seed(42) before generation?**

set_seed(42) makes the generation reproducible.

- Without a seed: each run gives a different text, because text generation includes randomness

- With a seed: your results stay consistent, easier for debugging and comparing outputs

C'est comme le random_state = 42, when we don't want our code to take idfferent values each time we run the code

**3. Other parameters that control generation**

- ***temperature***: cntrols how “creative” or random the model is.

--> low temperature (0.2): output is very predictable

--> high temperature (1.0+): output becomes more random

***top_k***: limits the model to choosing from the top k most likely tokens, this prevents it from picking extremely unlikely words

Ex: top_k = 50 → choose among the 50 best candidates

***do_sample***

- if do_sample=False, the model always takes the most likely next token (deterministic)

- if do_sample=True, it samples among possible tokens, so the output is more diverse

**4. Warning about truncation**

We can sometimes see: i"nput was truncated because it is too long...”

--> It means that GPT-2 has a maximum context length of 1024 tokens, if we input text longer, the beginning is cut off

So the model only keeps the last part of the input before generating

**5. Meaning of pad_token_id = eos_token_id**

GPT-2 does not have a padding token, because it was not trained for batch tasks originally

This tells the model: 'if you need to pad something, just use the end-of-sentence token”

We can explain it as a small hack so GPT-2 doesn’t crash

**6. Trade-offs between model size and generation quality**

***Larger GPT-2 models (medium, large, XL):***

- produce more coherent and meaningful text

- handle longer dependencies

- know more vocabulary patterns

--> BUT they have disadvantages:

- slower to run

- use more RAM/VRAM

- heavier to download

***Small models:***

- fast and lightweight

--> but text quality is sometimes less fluent, less consistent, or repetitive

So it depends on the task, we use small model for speed and large model for quality